# Demo: Reassemble Pipeline with Pre-Conversion Metrics

Same flow as `demo_reassemble_pipeline.ipynb`, plus a pre-conversion audit step. Searches for events, gathers the matches into a Roboto Collection, audits that collection with `training-dataset-metrics`, and invokes `roboto-to-lerobot-v2_1` against the same collection. The contract used for both audit and conversion is `contract_demo_reassemble.yaml`, stored in a dedicated Contracts dataset.

1. RoboQL search (default: `tags CONTAINS "to_lerobot"`).
2. Create an event collection from the matches (consumed by both actions).
3. Invoke `training-dataset-metrics` with `collection_id=<id>` to audit.
4. Invoke `roboto-to-lerobot-v2_1` with `collection_id=<id>` to convert.

## Initialization

In [ ]:
import roboto
from notebook_helpers import download_and_open_audit_report, tail_until_done

# Authentication (API token) is picked up from ~/.roboto/config.json
CONTRACTS_DATASET_ID = "ds_xxxxxxxxxxxx"  # your dataset that holds the contract YAML
CONTRACT = "contract_demo_reassemble.yaml"

roboto_client = roboto.RobotoClient.from_env()
roboto_search = roboto.RobotoSearch.for_roboto_client(roboto_client)

## Search Events and Build a Collection

Customize `EVENT_QUERY` below to whatever RoboQL expression selects the events you want included as episodes. The default — `tags CONTAINS "to_lerobot"` — picks up every event flagged for conversion in the org. You can refine with start-time floors, dataset IDs, metadata fields, etc. Examples:

- `tags CONTAINS "to_lerobot" AND start_time GREATER_THAN 1768343000000000000` — restrict to a recent batch.
- `tags CONTAINS "to_lerobot" AND dataset_id EQUALS "ds_xxxxxxxxxxxx"` — restrict to one source dataset.

Matched event IDs are gathered into a new Roboto Collection whose `resource_type` is `event`. Both the audit action and the conversion action receive this collection by ID, so the two runs are guaranteed to cover exactly the same episodes, and the converted dataset's manifest records `collection_id` and `collection_version` for reproducibility.

In [ ]:
import datetime

# Customize this RoboQL query to select the events you want to convert.
EVENT_QUERY = 'tags CONTAINS "to_lerobot"'

matched_events = list(roboto_search.find_events(EVENT_QUERY))
print(f"Found {len(matched_events)} event(s) matching the query")
assert matched_events, f"No events matched query: {EVENT_QUERY}"

collection = roboto.Collection.create(
    name=f"roboto-to-lerobot demo {datetime.datetime.now(datetime.UTC).isoformat(timespec='seconds')}",
    description=f"Events matched by RoboQL: {EVENT_QUERY!r}",
    resource_type=roboto.CollectionResourceType.Event,
    event_ids=[e.event_id for e in matched_events],
)
COLLECTION_ID = collection.collection_id
print(f"Created collection {COLLECTION_ID} (version {collection.record.version})")

## Create Output Dataset For LeRobot Training Data

We create the output dataset before running the audit so the HTML report can be uploaded directly into it, alongside the converted LeRobot data produced later.

In [ ]:
output_dataset = roboto.Dataset.create(
    name="Demo reassemble training dataset (with audit)",
    description=(
        f"LeRobot dataset for events in collection {COLLECTION_ID}; "
        f"includes pre-conversion audit report"
    ),
    tags=["le_robot_v2_1", "demo", "audited"],
    metadata={
        "source_collection_id": COLLECTION_ID,
        "source_collection_version": collection.record.version,
    },
)
print(f"LeRobot dataset to be saved in Roboto dataset '{output_dataset.dataset_id}'")

## Audit the Matched Events

Run `training-dataset-metrics` in pre-conversion mode against the collection built above. The action computes quality metrics against the raw topic streams and (with `write_audit_tags=True`) writes audit tags back onto each event: `audit:clean` when all checks pass, or one `audit:<flag>` per failed metric. Any prior `audit:*` tags are stripped first.

Passing `collection_id` puts the action in pre-conversion mode; it routes each event to its own source dataset for decoding, so the collection may span several datasets. `data_source_id` only has to supply the contract YAML, so we point it at the same Contracts dataset the conversion step uses. The HTML report is uploaded into `output_dataset` (created above) under `audit_reports/<invocation_id>/`.

In [ ]:
from roboto.domain import actions

metrics_action = actions.Action.from_name("training-dataset-metrics")
metrics_iv = metrics_action.invoke(
    invocation_source=actions.InvocationSource.Manual,
    data_source_id=CONTRACTS_DATASET_ID,
    input_data=[CONTRACT],
    upload_destination=actions.InvocationUploadDestination.dataset(
        output_dataset.dataset_id
    ),
    parameter_values={
        "collection_id": COLLECTION_ID,
        "contract": CONTRACT,
        "write_audit_tags": True,
    },
)

status = tail_until_done(metrics_iv)
assert status == actions.InvocationStatus.Completed, (
    f"Metrics invocation did not complete: {status}"
)
print(f"Metrics invocation: {metrics_iv.id}")

## Display the Audit Report

The action wrote `audit_reports/<invocation_id>/audit_report.html` into `output_dataset`, alongside the archived `contract.yaml` used for the run.

In [ ]:
download_and_open_audit_report(output_dataset, metrics_iv.id)

## Invoke `roboto-to-lerobot-v2_1`

We pass the collection ID created earlier; the action loads every event in it and converts each into one episode. The contract lives in its own Contracts dataset (`CONTRACTS_DATASET_ID`); we set that as `data_source_id` so the action can fetch the YAML by relative path. The output is written under `<invocation_id>/combined/` inside `output_dataset`, alongside a self-describing `manifest.json` that records `collection_id` and `collection_version` for reproducibility.

In [ ]:
roboto_to_lerobot = actions.Action.from_name("roboto-to-lerobot-v2_1")
convert_iv = roboto_to_lerobot.invoke(
    invocation_source=actions.InvocationSource.Manual,
    data_source_id=CONTRACTS_DATASET_ID,
    input_data=[CONTRACT],
    upload_destination=actions.InvocationUploadDestination.dataset(
        output_dataset.dataset_id
    ),
    parameter_values={
        "collection_id": COLLECTION_ID,
        "contract": CONTRACT,
    },
)

status = tail_until_done(convert_iv)
assert status == actions.InvocationStatus.Completed, (
    f"Conversion invocation did not complete: {status}"
)
print(f"Conversion invocation: {convert_iv.id}")